# Feature Engineering - House Price Prediction

This notebook applies the final EDA decisions and creates model-ready datasets.

Important rule:

We will not repeat EDA here.

EDA has already been completed in:

- `EDA_House_Price_clean_structured.ipynb`
- `Research/univariate/`
- `Research/bivariate/`
- `Research/multivariate/`
- `advanced_feature_engineering_route_house_price (1).md`

This notebook will only execute feature engineering decisions.

Main outputs:

- `fe_report/` → feature engineering reports, logs, validation summaries
- `fe_result/` → clean datasets, encoded datasets, submission-ready files

## 0. Project Setup

This section will:

- define project paths based on current repository structure
- load train/test data from `Data/`
- create `fe_report/` and `fe_result/`
- register EDA decision source folders
- prepare train/test/target variables

### Import libraries

In [23]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
import json
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
TARGET_COL = "SalePrice"
ID_COL = "Id"

### Define project paths from repository structure

In [27]:
from pathlib import Path

def find_project_root(start_path=None):
    """
    Find project root by searching current folder and parent folders
    for Data/train.csv and Data/test.csv.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = Path(start_path).resolve()

    candidate_roots = [start_path] + list(start_path.parents)

    for root in candidate_roots:
        data_dir = root / "Data"
        train_path = data_dir / "train.csv"
        test_path = data_dir / "test.csv"

        if train_path.exists() and test_path.exists():
            return root

    raise FileNotFoundError(
        "Could not find project root. Expected files: Data/train.csv and Data/test.csv"
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "Data"
RESEARCH_DIR = PROJECT_ROOT / "Research"

UNIVARIATE_DIR = RESEARCH_DIR / "univariate"
BIVARIATE_DIR = RESEARCH_DIR / "bivariate"
MULTIVARIATE_DIR = RESEARCH_DIR / "multivariate"

FE_REPORT_DIR = PROJECT_ROOT / "fe_report"
FE_RESULT_DIR = PROJECT_ROOT / "fe_result"

FE_STEP_REPORT_DIR = FE_REPORT_DIR / "step_reports"
FE_VALIDATION_DIR = FE_REPORT_DIR / "validation"
FE_PLOT_DIR = FE_REPORT_DIR / "plots"
FE_MODEL_DIR = FE_REPORT_DIR / "models"
FE_LOG_DIR = FE_REPORT_DIR / "logs"

for folder in [
    FE_REPORT_DIR,
    FE_RESULT_DIR,
    FE_STEP_REPORT_DIR,
    FE_VALIDATION_DIR,
    FE_PLOT_DIR,
    FE_MODEL_DIR,
    FE_LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Current working directory:", Path.cwd())
print("Detected PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("FE_REPORT_DIR:", FE_REPORT_DIR)
print("FE_RESULT_DIR:", FE_RESULT_DIR)

Current working directory: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\Research
Detected PROJECT_ROOT: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression
DATA_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\Data
FE_REPORT_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_report
FE_RESULT_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_result


### Verify required project files and folders

In [28]:
required_paths = {
    "Data folder": DATA_DIR,
    "Research folder": RESEARCH_DIR,
    "Bivariate reports folder": BIVARIATE_DIR,
    "Multivariate reports folder": MULTIVARIATE_DIR,
    "Train CSV": DATA_DIR / "train.csv",
    "Test CSV": DATA_DIR / "test.csv",
    "Data description": DATA_DIR / "data_description.txt",
    "Sample submission": DATA_DIR / "sample_submission.csv",
}

path_check_rows = []

for name, path in required_paths.items():
    path_check_rows.append({
        "item": name,
        "path": str(path),
        "exists": path.exists()
    })

project_path_check = pd.DataFrame(path_check_rows)

project_path_check.to_csv(
    FE_REPORT_DIR / "00_project_path_check.csv",
    index=False
)

project_path_check

,item,path,exists
0,Data folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
1,Research folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
2,Bivariate reports folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
3,Multivariate reports folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
4,Train CSV,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
5,Test CSV,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
6,Data description,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
7,Sample submission,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True


### Stop if required data files are missing

In [29]:
missing_required = project_path_check.loc[
    (project_path_check["item"].isin(["Train CSV", "Test CSV"])) &
    (project_path_check["exists"] == False)
]

if not missing_required.empty:
    display(missing_required)

    print("\nDebug: Current working directory contents:")
    for p in sorted(Path.cwd().iterdir()):
        print(p)

    raise FileNotFoundError(
        "Required train/test CSV files are missing. Check whether Data/train.csv and Data/test.csv exist."
    )

print("Required train/test files found.")

Required train/test files found.


In [30]:
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

sample_submission = None
if SAMPLE_SUBMISSION_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("train_raw shape:", train_raw.shape)
print("test_raw shape:", test_raw.shape)

if sample_submission is not None:
    print("sample_submission shape:", sample_submission.shape)

display(train_raw.head())
display(test_raw.head())

train_raw shape: (1460, 81)
test_raw shape: (1459, 80)
sample_submission shape: (1459, 2)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,6,1998,1998,Gable,CompShg,VinylSd,VinylSd,BrkFace,20.0,TA,TA,PConc,TA,TA,No,GLQ,602.0,Unf,0.0,324.0,926.0,GasA,Ex,Y,SBrkr,926,678,0,1604,0.0,0.0,2,1,3,1,Gd,7,Typ,1,Gd,Attchd,1998.0,Fin,2.0,470.0,TA,TA,Y,360,36,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1992,1992,Gable,CompShg,HdBoard,HdBoard,NaN,0.0,Gd,TA,PConc,Gd,TA,No,ALQ,263.0,Unf,0.0,1017.0,1280.0,GasA,Ex,Y,SBrkr,1280,0,0,1280,0.0,0.0,2,0,2,1,Gd,5,Typ,0,NaN,Attchd,1992.0,RFn,2.0,506.0,TA,TA,Y,0,82,0,0,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


### Basic data validation only

In [31]:
required_train_cols = [ID_COL, TARGET_COL]
required_test_cols = [ID_COL]

for col in required_train_cols:
    if col not in train_raw.columns:
        raise ValueError(f"Missing required train column: {col}")

for col in required_test_cols:
    if col not in test_raw.columns:
        raise ValueError(f"Missing required test column: {col}")

train_feature_cols = set(train_raw.columns) - {TARGET_COL}
test_feature_cols = set(test_raw.columns)

if train_feature_cols != test_feature_cols:
    print("Warning: Train/test feature columns are not perfectly aligned.")
    print("Train-only:", sorted(train_feature_cols - test_feature_cols))
    print("Test-only:", sorted(test_feature_cols - train_feature_cols))
else:
    print("Train/test feature columns are aligned.")

print("Basic validation completed.")

Train/test feature columns are aligned.
Basic validation completed.


### Separate IDs, target, and raw feature matrices

In [32]:
train_ids = train_raw[ID_COL].copy()
test_ids = test_raw[ID_COL].copy()

y_raw = train_raw[TARGET_COL].copy()
y_log = np.log1p(y_raw)

X_train_raw = train_raw.drop(columns=[TARGET_COL]).copy()
X_test_raw = test_raw.copy()

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_raw:", y_raw.shape)
print("y_log:", y_log.shape)

X_train_raw: (1460, 80)
X_test_raw: (1459, 80)
y_raw: (1460,)
y_log: (1460,)


### Register EDA decision sources

In [33]:
eda_decision_sources = {
    "main_eda_notebook": str(PROJECT_ROOT / "EDA_House_Price_clean_structured.ipynb"),
    "advanced_fe_route_markdown": str(PROJECT_ROOT / "advanced_feature_engineering_route_house_price (1).md"),
    "feature_engineering_markdown": str(PROJECT_ROOT / "Feature_Engineering.md"),
    "univariate_folder": str(UNIVARIATE_DIR),
    "bivariate_folder": str(BIVARIATE_DIR),
    "multivariate_folder": str(MULTIVARIATE_DIR),
    "data_description": str(DATA_DIR / "data_description.txt"),
}

eda_decision_sources_df = pd.DataFrame(
    [{"source_name": k, "path": v, "exists": Path(v).exists()} for k, v in eda_decision_sources.items()]
)

eda_decision_sources_df.to_csv(
    FE_REPORT_DIR / "00_eda_decision_sources.csv",
    index=False
)

eda_decision_sources_df

,source_name,path,exists
0,main_eda_notebook,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,False
1,advanced_fe_route_markdown,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
2,feature_engineering_markdown,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,False
3,univariate_folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
4,bivariate_folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
5,multivariate_folder,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
6,data_description,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True


### Create FE run metadata

In [34]:
fe_run_metadata = {
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "project_root": str(PROJECT_ROOT),
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "train_shape": train_raw.shape,
    "test_shape": test_raw.shape,
    "target_column": TARGET_COL,
    "id_column": ID_COL,
    "target_transform": "log1p",
    "random_state": RANDOM_STATE,
    "eda_repeat": False,
    "fe_report_dir": str(FE_REPORT_DIR),
    "fe_result_dir": str(FE_RESULT_DIR),
}

with open(FE_LOG_DIR / "00_fe_run_metadata.json", "w") as f:
    json.dump(fe_run_metadata, f, indent=4)

pd.DataFrame(
    [{"item": k, "value": str(v)} for k, v in fe_run_metadata.items()]
).to_csv(
    FE_REPORT_DIR / "00_fe_run_metadata.csv",
    index=False
)

fe_run_metadata

{'run_time': '2026-05-05 19:58:29',
 'project_root': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\House-Price-Prediction-Regression',
 'train_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\House-Price-Prediction-Regression\\Data\\train.csv',
 'test_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\House-Price-Prediction-Regression\\Data\\test.csv',
 'train_shape': (1460, 81),
 'test_shape': (1459, 80),
 'target_column': 'SalePrice',
 'id_column': 'Id',
 'target_transform': 'log1p',
 'random_state': 42,
 'eda_repeat': False,
 'fe_report_dir': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\House-Price-Prediction-Regression\\fe_report',
 'fe_result_dir': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\House-Price-Prediction-Regression\\fe_result'}

### Save raw snapshots for FE traceability

In [35]:
X_train_raw.to_csv(FE_RESULT_DIR / "00_X_train_raw.csv", index=False)
X_test_raw.to_csv(FE_RESULT_DIR / "00_X_test_raw.csv", index=False)

target_snapshot = pd.DataFrame({
    ID_COL: train_ids,
    TARGET_COL: y_raw,
    "LogSalePrice": y_log
})

target_snapshot.to_csv(FE_RESULT_DIR / "00_target_log_snapshot.csv", index=False)

print("Saved raw FE snapshots:")
print(FE_RESULT_DIR / "00_X_train_raw.csv")
print(FE_RESULT_DIR / "00_X_test_raw.csv")
print(FE_RESULT_DIR / "00_target_log_snapshot.csv")

Saved raw FE snapshots:
C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_result\00_X_train_raw.csv
C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_result\00_X_test_raw.csv
C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\House-Price-Prediction-Regression\fe_result\00_target_log_snapshot.csv


### Notebook setup decision

In [36]:
setup_decision = pd.DataFrame([
    {
        "decision_area": "EDA repeat",
        "decision": "No EDA will be repeated in Feature Engineering notebook.",
        "reason": "EDA already completed in Research folders and EDA notebook."
    },
    {
        "decision_area": "Data source",
        "decision": "Use train.csv and test.csv from Data/ folder.",
        "reason": "Matches current project repository structure."
    },
    {
        "decision_area": "Report output",
        "decision": "Save all FE reports inside fe_report/.",
        "reason": "Keeps FE outputs separate from EDA Research reports."
    },
    {
        "decision_area": "Dataset output",
        "decision": "Save all clean and engineered datasets inside fe_result/.",
        "reason": "Keeps model-ready outputs organized."
    },
    {
        "decision_area": "Target",
        "decision": "Use log1p(SalePrice) for modeling.",
        "reason": "RMSLE/log RMSE target."
    },
    {
        "decision_area": "Id",
        "decision": "Keep Id for tracking/submission, remove from model features later.",
        "reason": "Id is not predictive."
    }
])

setup_decision.to_csv(
    FE_STEP_REPORT_DIR / "00_setup_decision.csv",
    index=False
)

setup_decision

,decision_area,decision,reason
0,EDA repeat,No EDA will be repeated in Feature Engineering...,EDA already completed in Research folders and ...
1,Data source,Use train.csv and test.csv from Data/ folder.,Matches current project repository structure.
2,Report output,Save all FE reports inside fe_report/.,Keeps FE outputs separate from EDA Research re...
3,Dataset output,Save all clean and engineered datasets inside ...,Keeps model-ready outputs organized.
4,Target,Use log1p(SalePrice) for modeling.,RMSLE/log RMSE target.
5,Id,"Keep Id for tracking/submission, remove from m...",Id is not predictive.


### Setup Decision

Feature Engineering notebook setup is complete.

Decisions:

- Use `Data/train.csv` and `Data/test.csv`.
- Do not repeat EDA.
- Use previous EDA reports as decision source.
- Save all FE reports to `fe_report/`.
- Save all engineered datasets to `fe_result/`.
- Use `log1p(SalePrice)` as modeling target.
- Keep `Id` only for tracking and submission.

Next section will define the exact feature engineering decision constants from EDA.